In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [33]:
def zero_pad(X, pad, mode="constant", constant_values=(0, 0)):
    """
    Pad with zeros all images of the dataset X. The padding is applied to the height and width of an image.
    
    Argument:
    X -- python numpy array of shape (m, n_H, n_W, n_C) representing a batch of m images
    pad -- integer, amount of padding around each image on vertical and horizontal dimensions
    mode -- pad mode see: https://numpy.org/doc/stable/reference/generated/numpy.pad.html
    constant_values -- pad constant_values see: https://numpy.org/doc/stable/reference/generated/numpy.pad.html
    Returns:
    X_pad -- padded image of shape (m, n_H + 2 * pad, n_W + 2 * pad, n_C)
    """
    
    X_pad = np.pad(X, ((0, 0), (pad, pad), (pad, pad), (0, 0)), mode=mode, constant_values=constant_values)    
    
    return X_pad

In [40]:
A = abs(np.round(np.random.randn(1, 2, 3, 3), 2))
print(A.shape)
print(A)
A_pad = zero_pad(A, 2)
print(A_pad)

(1, 2, 3, 3)
[[[[0.12 0.7  1.32]
   [1.63 0.5  0.63]
   [0.71 1.63 0.24]]

  [[0.83 0.09 0.11]
   [0.93 1.28 1.33]
   [1.07 0.62 1.54]]]]
[[[[0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.   0.   0.  ]]

  [[0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.   0.   0.  ]]

  [[0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.12 0.7  1.32]
   [1.63 0.5  0.63]
   [0.71 1.63 0.24]
   [0.   0.   0.  ]
   [0.   0.   0.  ]]

  [[0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.83 0.09 0.11]
   [0.93 1.28 1.33]
   [1.07 0.62 1.54]
   [0.   0.   0.  ]
   [0.   0.   0.  ]]

  [[0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.   0.   0.  ]]

  [[0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.   0.   0.  ]
   [0.   0.   0.  ]]]]


In [15]:
def conv_single_step(a_slice_prev, W, b):
    """
    Apply one filter defined by parameters W on a single slice (a_slice_prev) of the output activation 
    of the previous layer.
    
    Arguments:
    a_slice_prev -- slice of input data of shape (f, f, n_C_prev)
    W -- Weight parameters contained in a window - matrix of shape (f, f, n_C_prev)
    b -- Bias parameters contained in a window - matrix of shape (1, 1, 1)
    
    Returns:
    Z -- a scalar value, the result of convolving the sliding window (W, b) on a slice x of the input data
    """

    s = np.multiply(a_slice_prev, W)
    Z = np.sum(s)
    Z = Z + float(b)
    
    return Z

In [39]:
A = abs(np.round(np.random.randn(2, 4, 4), 2))
W = abs(np.round(np.random.randn(2, 4, 4), 2))
b = abs(np.round(np.random.randn(1, 1, 1), 2))
print("A")
print(A)
print("W")
print(W)
print("b")
print(b)
print("Z")
Z = conv_single_step(A, W, b)
print(Z)

A
[[[0.1  2.21 0.8  1.09]
  [0.42 1.22 0.93 1.22]
  [1.93 1.29 0.17 0.58]
  [0.65 1.83 1.17 0.33]]

 [[0.34 0.82 1.16 0.27]
  [0.23 2.03 0.19 0.74]
  [0.82 1.7  0.27 0.4 ]
  [0.52 0.07 0.8  0.93]]]
W
[[[0.01 1.14 1.08 0.58]
  [0.75 1.6  0.64 0.09]
  [1.77 1.06 0.77 0.07]
  [0.81 1.25 1.38 0.02]]

 [[0.41 1.09 0.21 0.49]
  [1.03 0.99 0.92 1.08]
  [1.04 1.45 0.11 0.68]
  [1.41 0.54 0.84 1.39]]]
b
[[[0.71]]]
Z
28.0737


/tmp/ipykernel_5338/4159457354.py:17: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  Z = Z + float(b)


In [41]:
def conv_forward(A_prev, W, b, hparameters):
    """
    Implements the forward propagation for a convolution function
    
    Arguments:
    A_prev -- output activations of the previous layer, 
        numpy array of shape (m, n_H_prev, n_W_prev, n_C_prev)
    W -- Weights, numpy array of shape (f, f, n_C_prev, n_C)
    b -- Biases, numpy array of shape (1, 1, 1, n_C)
    hparameters -- python dictionary containing "stride" and "pad"
        
    Returns:
    Z -- conv output, numpy array of shape (m, n_H, n_W, n_C)
    cache -- cache of values needed for the conv_backward() function
    """
    
    (m, n_H_prev, n_W_prev, n_C_prev) = A_prev.shape
    (f, f, n_C_prev, n_C) = W.shape
    stride = hparameters["stride"]
    pad = hparameters["pad"]
    n_H = int(((n_H_prev - f + (2*pad))/stride) + 1)
    n_W = int(((n_W_prev - f + (2*pad))/stride) + 1)
    
    Z = np.zeros((m, n_H, n_W, n_C))
    A_prev_pad = zero_pad(A_prev, pad)
    
    for i in range(m):
        a_prev_pad = A_prev_pad[i]
        for h in range(n_H):  
            vert_start = h * stride
            vert_end = vert_start + f
            for w in range(n_W):  
                horiz_start = w * stride
                horiz_end = horiz_start + f
                for c in range(n_C):
                    a_slice_prev = a_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :]
                    weights = W[:, :, :, c]
                    biases = b[:, :, :, c]
                    Z[i, h, w, c] = conv_single_step(a_slice_prev, weights, biases)
    cache = (A_prev, W, b, hparameters)
    
    return Z, cache

In [51]:
A_prev = np.random.randn(2, 5, 7, 4)
W = np.random.randn(3, 3, 4, 8)
b = np.random.randn(1, 1, 1, 8)
hparameters = {"pad" : 1,
               "stride": 2}

Z, cache_conv = conv_forward(A_prev, W, b, hparameters)
print(Z)
print(A_prev.shape)
print(Z.shape)


[[[[ -5.03822128  -0.38748626   4.34215523   1.33670444   1.67479635
      8.08071353  -0.51552921   9.93928724]
   [ -8.83349009  -3.12770772   1.49367625   6.13554567   0.37048799
     -2.09879539   0.1887776   -1.21283412]
   [ -1.36519284   5.19967737   3.4828166    3.78561011   0.19767155
     -3.56628751  -2.3215654    1.38436649]
   [  0.28728777   3.78440991  -4.47854295  -5.34913886   2.82201183
     -4.53699623   2.66866152  -0.65874535]]

  [[  1.56148297   2.34432949  -5.12478482   3.52158968  -5.47005237
      1.39682659   4.5383858   -6.91098796]
   [  3.90436975  -1.90313135  -1.08588592   8.27144067  -2.84008372
     -9.67364022  11.61574911 -13.79127593]
   [ -1.41605095   3.30534419   4.60536489  -2.62541152  -4.46161842
      2.38520632  -0.654375     5.14778973]
   [  3.89916228   5.33071708  -6.12619538  -2.3517406   -2.60375254
     -8.74969207   2.65042207  -2.73901547]]

  [[  2.32946294   7.08007476  -3.60613221   5.21794128  -1.3188637
      2.63466875   0.613

/tmp/ipykernel_5338/4159457354.py:17: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  Z = Z + float(b)


In [43]:
def pool_forward(A_prev, hparameters, mode = "max"):
    """
    Implements the forward pass of the pooling layer
    
    Arguments:
    A_prev -- Input data, numpy array of shape (m, n_H_prev, n_W_prev, n_C_prev)
    hparameters -- python dictionary containing "f" and "stride"
    mode -- the pooling mode you would like to use, defined as a string ("max" or "average")
    
    Returns:
    A -- output of the pool layer, a numpy array of shape (m, n_H, n_W, n_C)
    cache -- cache used in the backward pass of the pooling layer, contains the input and hparameters 
    """
    
    
    (m, n_H_prev, n_W_prev, n_C_prev) = A_prev.shape
    
    f = hparameters["f"]
    stride = hparameters["stride"]
    
    n_H = int(1 + (n_H_prev - f) / stride)
    n_W = int(1 + (n_W_prev - f) / stride)
    n_C = n_C_prev
    
    A = np.zeros((m, n_H, n_W, n_C))              
    
   
    for i in range(m):                         
        for h in range(n_H):                  
            vert_start = h * stride
            vert_end = vert_start + f
            for w in range(n_W):              
                horiz_start = w * stride
                horiz_end = horiz_start + f
                for c in range(n_C):          
                    a_prev_slice = A_prev[i, vert_start:vert_end, horiz_start:horiz_end, c]
                    if mode == "max":
                        A[i, h, w, c] = np.max(a_prev_slice)
                    elif mode == "average":
                        A[i, h, w, c] = np.mean(a_prev_slice)
    
   
    
 
    cache = (A_prev, hparameters)
    
    return A, cache

In [45]:
A = A_prev = np.random.randn(2, 5, 7, 4)
hparameters = {
    "pad" : 1,
    "stride": 2,
    "f": 3
}

A_shrinked, cache = pool_forward(A_prev, hparameters, mode = "max")
print("A")
print(A.shape)
print(A)
print("A_shrinked")
print(A_shrinked.shape)
print(A_shrinked)

A
(2, 5, 7, 4)
[[[[-1.07421265e-01 -5.00968831e-01 -1.66887535e+00 -6.39777662e-01]
   [ 9.32332057e-01  2.31482938e-01  7.81361636e-01 -5.22031771e-01]
   [ 1.01967168e+00 -1.91321429e-01  8.15023200e-02  9.98067257e-01]
   [ 4.09093105e-02 -5.59096056e-01  4.68190455e-01  2.05046475e+00]
   [-1.28904689e+00 -1.57087346e+00  8.57253851e-01  2.06865442e-01]
   [ 3.04308105e-02 -2.37598428e-02  2.21667011e+00  1.48821121e+00]
   [ 1.27659312e-02 -9.22336928e-01  6.62811343e-01  1.55750250e+00]]

  [[-9.43235412e-02 -6.86587157e-01  7.40757312e-01 -9.46888742e-01]
   [-4.34886871e-02 -3.13817109e-01 -3.23335789e-01 -1.55239809e+00]
   [ 8.37784197e-01  1.00292246e+00 -4.31623572e-01  1.54392456e+00]
   [-9.06187838e-01  1.81842482e+00 -7.18722483e-01  3.89612621e+00]
   [-1.12972659e-01 -1.21538873e+00 -2.04411031e+00  1.45183403e+00]
   [ 9.14521433e-01 -8.36344152e-01 -1.56839669e+00 -1.25000764e+00]
   [ 4.12001857e-01 -6.08938790e-01 -5.78846227e-02  6.40151256e-01]]

  [[-1.36452551

In [46]:
def conv_backward(dZ, cache):
    """
    Implement the backward propagation for a convolution function
    
    Arguments:
    dZ -- gradient of the cost with respect to the output of the conv layer (Z), numpy array of shape (m, n_H, n_W, n_C)
    cache -- cache of values needed for the conv_backward(), output of conv_forward()
    
    Returns:
    dA_prev -- gradient of the cost with respect to the input of the conv layer (A_prev),
               numpy array of shape (m, n_H_prev, n_W_prev, n_C_prev)
    dW -- gradient of the cost with respect to the weights of the conv layer (W)
          numpy array of shape (f, f, n_C_prev, n_C)
    db -- gradient of the cost with respect to the biases of the conv layer (b)
          numpy array of shape (1, 1, 1, n_C)
    """    
    (A_prev, W, b, hparameters) = cache
    (m, n_H_prev, n_W_prev, n_C_prev) = A_prev.shape
    (f, f, n_C_prev, n_C) = W.shape

    stride = hparameters["stride"]
    pad = hparameters["pad"]
    (m, n_H, n_W, n_C) = dZ.shape
    
    dA_prev = np.zeros(A_prev.shape)                          
    dW = np.zeros(W.shape)
    db = np.zeros(b.shape)
    
    A_prev_pad = zero_pad(A_prev, pad)
    dA_prev_pad = zero_pad(dA_prev, pad)
    
    for i in range(m):
        
        a_prev_pad = A_prev_pad[i]
        da_prev_pad = dA_prev_pad[i]
        
        for h in range(n_H):
            for w in range(n_W):               
                for c in range(n_C):
                    
                    vert_start = h * stride
                    vert_end = vert_start + f
                    horiz_start = w * stride
                    horiz_end = horiz_start + f
                    
                    a_slice = a_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :]
                    
                    da_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :] += W[:,:,:,c] * dZ[i, h, w, c]
                    dW[:,:,:,c] += a_slice * dZ[i, h, w, c]
                    db[:,:,:,c] += dZ[i, h, w, c]
    
    dA_prev = dA_prev_pad[:, pad:-pad, pad:-pad, :]
    
    return dA_prev, dW, db

In [47]:
A_prev = np.random.randn(10, 4, 4, 3)
W = np.random.randn(2, 2, 3, 8)
b = np.random.randn(1, 1, 1, 8)
hparameters = {"pad" : 2,
               "stride": 2}

Z, cache_conv = conv_forward(A_prev, W, b, hparameters)
print(Z)
dA, dW, db = conv_backward(Z, cache_conv)
print(dA)


[[[[  0.83685398   0.17806203  -0.44920307 ...   1.07778795
      0.63768924  -0.71863147]
   [  0.83685398   0.17806203  -0.44920307 ...   1.07778795
      0.63768924  -0.71863147]
   [  0.83685398   0.17806203  -0.44920307 ...   1.07778795
      0.63768924  -0.71863147]
   [  0.83685398   0.17806203  -0.44920307 ...   1.07778795
      0.63768924  -0.71863147]]

  [[  0.83685398   0.17806203  -0.44920307 ...   1.07778795
      0.63768924  -0.71863147]
   [  0.16591678   8.20874182  -4.83414631 ...  10.17806294
     -5.51947999   3.49538399]
   [  0.54135898   0.06500512   0.16897727 ...  -0.62389311
      1.34614198  -2.69996672]
   [  0.83685398   0.17806203  -0.44920307 ...   1.07778795
      0.63768924  -0.71863147]]

  [[  0.83685398   0.17806203  -0.44920307 ...   1.07778795
      0.63768924  -0.71863147]
   [ -1.3578106   -2.5280202   -2.107827   ...   1.43584225
     -3.1395977    1.19089365]
   [  0.90254406  -5.26058622   2.50658722 ...   1.39961345
     -0.71869091  -0.44505

/tmp/ipykernel_5338/4159457354.py:17: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  Z = Z + float(b)


In [48]:
def create_mask_from_window(x):
    """
    Creates a mask from an input matrix x, to identify the max entry of x.
    
    Arguments:
    x -- Array of shape (f, f)
    
    Returns:
    mask -- Array of the same shape as window, contains a True at the position corresponding to the max entry of x.
    """    
    max_val = np.max(x)
    mask = (x == max_val)
 
    return mask

In [49]:
def distribute_value(dz, shape):
    """
    Distributes the input value in the matrix of dimension shape
    
    Arguments:
    dz -- input scalar
    shape -- the shape (n_H, n_W) of the output matrix for which we want to distribute the value of dz
    
    Returns:
    a -- Array of size (n_H, n_W) for which we distributed the value of dz
    """    

    (n_H, n_W) = shape
    average = dz / (n_H * n_W)
    a = np.ones(shape) * average
    
    return a

In [50]:
def pool_backward(dA, cache, mode="max"):
    """
    Implements the backward pass of the pooling layer

    Arguments:
    dA -- gradient of cost with respect to the output of the pooling layer, same shape as A
    cache -- cache output from the forward pass of the pooling layer, contains the layer's input and hparameters
    mode -- the pooling mode you would like to use, defined as a string ("max" or "average")

    Returns:
    dA_prev -- gradient of cost with respect to the input of the pooling layer, same shape as A_prev
    """

    (A_prev, hparameters) = cache
    stride = hparameters["stride"]
    f = hparameters["f"]

    m, n_H_prev, n_W_prev, n_C_prev = A_prev.shape
    m, n_H, n_W, n_C = dA.shape

    dA_prev = np.zeros(A_prev.shape)

    for i in range(m):
        a_prev = A_prev[i]
        for h in range(n_H):  
            for w in range(n_W):  
                for c in range(n_C):  

                    vert_start = h * stride
                    vert_end = vert_start + f
                    horiz_start = w * stride
                    horiz_end = horiz_start + f

                    if mode == "max":
                        a_prev_slice = a_prev[vert_start: vert_end, horiz_start: horiz_end, c]
                        mask = (a_prev_slice == np.max(a_prev_slice))
                        dA_prev[i, vert_start: vert_end, horiz_start: horiz_end, c] += mask * dA[i, h, w, c]

                    elif mode == "average":
                        da = dA[i, h, w, c]
                        shape = (f, f)
                        dA_prev[i, vert_start: vert_end, horiz_start: horiz_end, c] += distribute_value(da, shape)

    return dA_prev
